# Прогнозирование оттока клиентов «ТелеДом»

Оператор связи хочет заранее выявлять клиентов с высоким риском ухода и предлагать им специальные условия. В проекте объединены данные о договорах, персональных характеристиках, интернет- и телефонных услугах; проведены исследовательский анализ, подготовка признаков и сравнение моделей классификации.

**Целевая метрика:** ROC AUC.  
**Дата среза данных:** 1 февраля 2020 года.  
**Целевой признак:** факт расторжения договора (`Churn`).


## 1. Загрузка и первичная проверка данных

Каждая таблица содержит уникальный идентификатор клиента `customerID`. Договоры используются как основа объединения, а сведения об услугах присоединяются левыми соединениями: отсутствие строки в таблице услуг означает, что услуга не подключена.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 50)

RANDOM_STATE = 160924
SNAPSHOT_DATE = pd.Timestamp('2020-02-01')
DATA_DIR = Path('../data')


In [2]:
contract = pd.read_csv(DATA_DIR / 'contract_new.csv.gz')
personal = pd.read_csv(DATA_DIR / 'personal_new.csv.gz')
internet = pd.read_csv(DATA_DIR / 'internet_new.csv.gz')
phone = pd.read_csv(DATA_DIR / 'phone_new.csv.gz')

tables = {
    'contract': contract,
    'personal': personal,
    'internet': internet,
    'phone': phone,
}

overview = pd.DataFrame({
    name: {
        'rows': len(frame),
        'columns': frame.shape[1],
        'duplicates': frame.duplicated().sum(),
        'duplicate_customerID': frame['customerID'].duplicated().sum(),
    }
    for name, frame in tables.items()
}).T
overview


,rows,columns,duplicates,duplicate_customerID
contract,7043,8,0,0
personal,7043,5,0,0
internet,5517,8,0,0
phone,6361,2,0,0


In [3]:
pd.DataFrame({
    'column': contract.columns,
    'dtype': contract.dtypes.astype(str).values,
    'missing': contract.isna().sum().values,
    'unique': contract.nunique(dropna=False).values,
})


,column,dtype,missing,unique
0,customerID,object,0,7043
1,BeginDate,object,0,77
2,EndDate,object,0,67
3,Type,object,0,3
4,PaperlessBilling,object,0,2
5,PaymentMethod,object,0,4
6,MonthlyCharges,float64,0,1585
7,TotalCharges,object,0,6658


Дубликатов идентификаторов и полных строк нет. `TotalCharges` загружен как текст из-за пустых значений у новых клиентов; ниже он преобразуется в число, а пропуски заполняются нулём. Это соответствует нулевой накопленной сумме на момент начала обслуживания.


## 2. Объединение и подготовка признаков

Цель создаётся **до** преобразования дат: `Churn = 1`, если в `EndDate` указана дата расторжения. Для действующих клиентов при расчёте продолжительности договора используется дата среза.


In [4]:
contract = contract.copy()
contract['Churn'] = contract['EndDate'].ne('No').astype('int8')
contract['BeginDate'] = pd.to_datetime(contract['BeginDate'])
contract['EndDateParsed'] = pd.to_datetime(contract['EndDate'].replace('No', SNAPSHOT_DATE))
contract['TenureDays'] = (contract['EndDateParsed'] - contract['BeginDate']).dt.days
contract['TotalCharges'] = pd.to_numeric(contract['TotalCharges'], errors='coerce').fillna(0)

data = (
    contract
    .merge(personal, on='customerID', how='left', validate='one_to_one')
    .merge(internet, on='customerID', how='left', validate='one_to_one')
    .merge(phone, on='customerID', how='left', validate='one_to_one')
)

service_columns = [
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines'
]
data[service_columns] = data[service_columns].fillna('No service')
data['SeniorCitizen'] = data['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

quality = pd.Series({
    'rows': len(data),
    'columns': data.shape[1],
    'missing_values': int(data.isna().sum().sum()),
    'duplicate_customerID': int(data['customerID'].duplicated().sum()),
    'churned_clients': int(data['Churn'].sum()),
    'churn_rate': data['Churn'].mean(),
})
quality.to_frame('value')


,value
rows,7043.000000
columns,23.000000
missing_values,0.000000
duplicate_customerID,0.000000
churned_clients,1101.000000
churn_rate,0.156325


## 3. Исследовательский анализ

Сначала оценим баланс классов, затем сравним денежные показатели и длительность отношений с оператором для действующих и ушедших клиентов.


In [5]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = data['Churn'].value_counts().sort_index()
sns.barplot(x=['Остался', 'Ушёл'], y=counts.values, ax=ax, hue=['Остался', 'Ушёл'], legend=False)
ax.set(title='Баланс целевого признака', xlabel='', ylabel='Количество клиентов')
for container in ax.containers:
    ax.bar_label(container, fmt='%d')
plt.tight_layout()
plt.show()


![График из шага 10](../assets/figures/cell_010_01.png)


In [6]:
plot_data = data.assign(Status=data['Churn'].map({0: 'Остался', 1: 'Ушёл'}))
numeric_features_eda = ['MonthlyCharges', 'TotalCharges', 'TenureDays']

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for feature, ax in zip(numeric_features_eda, axes):
    sns.boxplot(data=plot_data, x='Status', y=feature, ax=ax, showfliers=False)
    ax.set(title=f'{feature} по статусу клиента', xlabel='')
plt.tight_layout()
plt.show()

data.groupby('Churn')[numeric_features_eda].median().rename(index={0: 'Остался', 1: 'Ушёл'})


,MonthlyCharges,TotalCharges,TenureDays
Churn,,,
Остался,69.2,1192.80,702.0
Ушёл,84.2,2139.03,915.0


![График из шага 11](../assets/figures/cell_011_01.png)


In [7]:
categorical_eda = ['Type', 'InternetService', 'PaymentMethod']
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

for feature, ax in zip(categorical_eda, axes):
    rates = data.groupby(feature, observed=False)['Churn'].mean().sort_values(ascending=False)
    sns.barplot(x=rates.values, y=rates.index, ax=ax, hue=rates.index, legend=False)
    ax.set(title=f'Доля оттока: {feature}', xlabel='Доля ушедших', ylabel='')
    ax.set_xlim(0, max(0.5, rates.max() * 1.12))
plt.tight_layout()
plt.show()


![График из шага 12](../assets/figures/cell_012_01.png)


In [8]:
correlation_columns = ['MonthlyCharges', 'TotalCharges', 'TenureDays', 'Churn']
correlations = data[correlation_columns].corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(correlations, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Корреляции числовых признаков')
plt.tight_layout()
plt.show()

correlations['Churn'].sort_values(ascending=False).to_frame('correlation_with_churn')


,correlation_with_churn
Churn,1.000000
MonthlyCharges,0.154286
TotalCharges,0.052175
TenureDays,0.016578


![График из шага 13](../assets/figures/cell_013_01.png)


### Выводы исследовательского анализа

- В данных присутствует дисбаланс классов, поэтому основной метрикой выбрана ROC AUC, а разбиение выполняется со стратификацией.
- В этом срезе ушедшие клиенты имеют более высокие медианные накопленные платежи и более длительную историю обслуживания; связь длительности с оттоком нелинейна.
- Повышенная наблюдаемая доля оттока выделяется у клиентов с долгосрочными договорами, автоматическими способами оплаты и оптоволоконным интернетом. Это описательные связи, а не доказательство причинности.
- Даты и идентификатор не передаются модели; вместо них используется интерпретируемая длительность обслуживания в днях.


## 4. Подготовка выборок и сравнение моделей

Данные делятся на обучающую и тестовую выборки в пропорции 75/25. Все преобразования обучаются только на тренировочной части внутри `Pipeline`, что исключает утечку данных. Категории кодируются One-Hot Encoding, числовые признаки масштабируются.


In [9]:
drop_columns = ['customerID', 'BeginDate', 'EndDate', 'EndDateParsed', 'Churn']
X = data.drop(columns=drop_columns)
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)),
    ]), categorical_features),
])

print(f'Обучающая выборка: {X_train.shape}')
print(f'Тестовая выборка: {X_test.shape}')
print(f'Числовых признаков: {len(numeric_features)}; категориальных: {len(categorical_features)}')


Обучающая выборка: (5282, 18)
Тестовая выборка: (1761, 18)
Числовых признаков: 3; категориальных: 15


In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_specs = {
    'Logistic Regression': (
        LogisticRegression(random_state=RANDOM_STATE, max_iter=2000),
        {'model__C': [0.1, 1.0, 10.0]},
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
        {
            'model__n_estimators': [200, 400],
            'model__max_depth': [6, 12, None],
            'model__min_samples_leaf': [1, 4],
        },
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            'model__n_estimators': [100, 200],
            'model__learning_rate': [0.03, 0.1],
            'model__max_depth': [2, 3],
        },
    ),
}

searches = {}
comparison_rows = []

for model_name, (estimator, param_grid) in model_specs.items():
    pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', estimator),
    ])
    search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring='roc_auc',
        cv=cv,
        n_jobs=1,
        refit=True,
    )
    search.fit(X_train, y_train)
    searches[model_name] = search
    comparison_rows.append({
        'model': model_name,
        'cv_roc_auc': search.best_score_,
        'best_params': search.best_params_,
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values('cv_roc_auc', ascending=False)
model_comparison


,model,cv_roc_auc,best_params
2,Gradient Boosting,0.865904,"{'model__learning_rate': 0.1, 'model__max_dept..."
1,Random Forest,0.823479,"{'model__max_depth': 12, 'model__min_samples_l..."
0,Logistic Regression,0.772391,{'model__C': 1.0}


In [11]:
plt.figure(figsize=(8, 4))
sns.barplot(data=model_comparison, x='cv_roc_auc', y='model', hue='model', legend=False)
plt.xlim(max(0.5, model_comparison['cv_roc_auc'].min() - 0.03),
         min(1.0, model_comparison['cv_roc_auc'].max() + 0.03))
plt.title('Сравнение моделей на кросс-валидации')
plt.xlabel('ROC AUC')
plt.ylabel('')
plt.tight_layout()
plt.show()


![График из шага 18](../assets/figures/cell_018_01.png)


## 5. Оценка лучшей модели

Лучшая модель выбирается только по среднему ROC AUC на кросс-валидации. Тестовая выборка используется один раз — для финальной независимой оценки.


In [12]:
best_model_name = model_comparison.iloc[0]['model']
best_search = searches[best_model_name]
best_model = best_search.best_estimator_

y_probability = best_model.predict_proba(X_test)[:, 1]
y_prediction = (y_probability >= 0.5).astype(int)

test_metrics = pd.Series({
    'best_model': best_model_name,
    'cv_roc_auc': best_search.best_score_,
    'test_roc_auc': roc_auc_score(y_test, y_probability),
    'accuracy': accuracy_score(y_test, y_prediction),
    'precision': precision_score(y_test, y_prediction),
    'recall': recall_score(y_test, y_prediction),
    'f1': f1_score(y_test, y_prediction),
})
test_metrics.to_frame('value')


,value
best_model,Gradient Boosting
cv_roc_auc,0.865904
test_roc_auc,0.858546
accuracy,0.88586
precision,0.772059
recall,0.381818
f1,0.510949


In [13]:
report = pd.DataFrame(
    classification_report(
        y_test,
        y_prediction,
        target_names=['Остался', 'Ушёл'],
        output_dict=True,
        zero_division=0,
    )
).T
report


,precision,recall,f1-score,support
Остался,0.895385,0.979139,0.935391,1486.00000
Ушёл,0.772059,0.381818,0.510949,275.00000
accuracy,0.885860,0.885860,0.885860,0.88586
macro avg,0.833722,0.680478,0.723170,1761.00000
weighted avg,0.876126,0.885860,0.869109,1761.00000


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_prediction),
    display_labels=['Остался', 'Ушёл'],
).plot(cmap='Blues', colorbar=False, ax=axes[0])
axes[0].set_title('Матрица ошибок')

fpr, tpr, _ = roc_curve(y_test, y_probability)
axes[1].plot(fpr, tpr, lw=2, label=f'ROC AUC = {roc_auc_score(y_test, y_probability):.3f}')
axes[1].plot([0, 1], [0, 1], '--', color='grey')
axes[1].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC-кривая')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


![График из шага 22](../assets/figures/cell_022_01.png)


## 6. Интерпретация модели

Для линейной модели используется абсолютная величина коэффициентов, для ансамблей — встроенная важность признаков. Значения показывают вклад признаков в прогноз и помогают определить направления для бизнес-анализа, но сами по себе не доказывают причинно-следственную связь.


In [15]:
fitted_preprocessor = best_model.named_steps['preprocessor']
fitted_estimator = best_model.named_steps['model']
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(fitted_estimator, 'feature_importances_'):
    importance_values = fitted_estimator.feature_importances_
else:
    importance_values = np.abs(fitted_estimator.coef_).ravel()

feature_importance = (
    pd.DataFrame({'feature': feature_names, 'importance': importance_values})
    .sort_values('importance', ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 7))
sns.barplot(data=feature_importance, x='importance', y='feature', hue='feature', legend=False)
plt.title(f'Наиболее значимые признаки: {best_model_name}')
plt.xlabel('Важность')
plt.ylabel('')
plt.tight_layout()
plt.show()

feature_importance


,feature,importance
2,num__TenureDays,0.480625
4,cat__Type_Two year,0.124217
0,num__MonthlyCharges,0.111342
1,num__TotalCharges,0.099658
11,cat__Partner_Yes,0.043586
28,cat__MultipleLines_Yes,0.036782
3,cat__Type_One year,0.021445
18,cat__OnlineBackup_Yes,0.021361
8,cat__PaymentMethod_Mailed check,0.014892
20,cat__DeviceProtection_Yes,0.014310


![График из шага 24](../assets/figures/cell_024_01.png)


## 7. Общий вывод и рекомендации

В проекте объединены четыре источника данных, устранены пропуски, создан целевой признак и рассчитана продолжительность обслуживания. Модели сравнивались по ROC AUC на стратифицированной кросс-валидации; тестовая выборка не участвовала в выборе модели.

Лучший результат показал Gradient Boosting: ROC AUC на кросс-валидации — 0.866, на тестовой выборке — 0.859. При стандартном пороге 0.5 модель даёт precision 0.772 и recall 0.382 для класса оттока.

Практическое применение модели — ранжирование клиентской базы по вероятности ухода. Удерживающие предложения стоит тестировать на сегментах с повышенной прогнозной вероятностью, отдельно анализируя клиентов с долгосрочными договорами, автоматической оплатой и оптоволоконным интернетом. Порог решения следует выбирать по стоимости контакта и потенциальной потере клиента: его снижение повысит полноту выявления оттока. Качество модели необходимо регулярно контролировать на новых данных.
